# 河北古塔 43 样本全量主实验 / Global analysis of 43 Hebei pagodas

## 使用说明 / How to use

**解压完整代码包 → 打开本 notebook → 选择“运行全部 / Run All”。无需手动安装本项目、修改路径、选择数据、设置参数或输入命令。**

**Extract the complete archive → open this notebook → select Run All. No manual project installation, path editing, data selection, parameter changes, or terminal commands are required.**

本 notebook 完整复现论文全量主实验（原 3.2.1）。每个模块各调用一个函数，各阶段显示全部样本或相应分析图；算法、循环和绘图细节封装在库内。已包含 43 份 0.009H 点云及对应 k=7 法向，共 86 个 NPY 文件。

This notebook reproduces the complete primary global experiment (former Section 3.2.1). Each module has its own function call and figures; implementation details stay in the library. All 43 point clouds at 0.009H and their matching k=7 normal arrays are included (86 NPY files).

**运行条件 / Prerequisites:** 使用 Python 3.10 或以上的 Jupyter/VS Code notebook 内核。启动单元格会自动检查依赖；缺少依赖时自动安装，此时需要联网且当前环境可写。依赖齐全时可离线运行。无需原始 OBJ、Open3D 或 GPU。

Use a Python 3.10+ notebook kernel in Jupyter or VS Code. Startup automatically checks dependencies and installs missing libraries; installation requires internet and a writable environment. With dependencies present, execution can be offline. No raw OBJ, Open3D, or GPU is needed.

保持解压后的文件结构完整。每次“运行全部”会自动生成新的 `results/global43_时间戳/` 目录，不覆盖以前的结果。

Keep the extracted directory intact. Every Run All generates a new `results/global43_timestamp/` directory and preserves previous runs.

| 参数 / Parameter | 固定设置 / Setting | 含义 / Meaning |
| --- | --- | --- |
| 体素系数 / Voxel coefficient | 0.009H | H 是原模型的竖直高度；输入已体素化 / Original mesh height; already voxelized inputs |
| 法向邻域 / Normal neighborhood | k=7，含自身 / including self | 已计算并矫正朝向的 PCA 法向 / Prepared, centroid-oriented PCA normals |
| 角格 / Angular bins | 1° × 1° | 每个样本 180×360，行极角、列方位 / Polar rows, azimuth columns |
| 峰值参考 / Peak reference | θ∈[80°,100°)，五列找峰 / five-column peak search | 最大方位峰移动到0° / Shift peak to 0° |
| 方位平滑 / Azimuth smoothing | 1×5，循环重叠、步长1 / circular, stride one | 对齐后平滑，矩阵尺寸不变 / After alignment, no decimation |
| 分布距离 / Distribution distance | √JSD，底数2 / base 2 | SciPy 函数已开方 / SciPy already returns the square root |
| 聚类 / Clustering | 平均链接 / Average linkage | k=2…8中按轮廓系数选簇 / Silhouette selection over valid cuts |


## 0. 自动准备 / Automatic setup

自动定位包内源代码、检查/安装 NumPy、SciPy 与 Matplotlib、核对 86 个输入文件的 SHA256、启用内嵌图像并确定输出目录。无需编辑这个代码块。

Automatically locate bundled source, check/install NumPy/SciPy/Matplotlib, verify all 86 SHA256 hashes, enable inline figures, and select the output directory. Do not edit this cell.

In [ ]:
# 中文：从 notebook 所在目录、其上级或默认解压目录自动找到启动文件。
# English: locate startup from the notebook directory, its parents, or the extracted folder.
from pathlib import Path
import runpy
_setup_file = next(
    folder / "notebook_setup.py"
    for base in (Path.cwd(), *Path.cwd().parents)
    for folder in (base, base / "nvpd_global_code")
    if (folder / "notebook_setup.py").is_file() and (folder / "src/nvpd_global").is_dir()
)
# 中文：当前内核自动准备；不需要执行 pip install . 或填写任何路径。
# English: prepare this kernel automatically; no pip install . or path configuration.
environment = runpy.run_path(str(_setup_file))["prepare_notebook"](_setup_file.parent)
from nvpd_global import (preprocess, compute_nvpd, align_nvpd, smooth_nvpd,
                         compute_distances, cluster_hca, compare_clusters, save_results)


## 1. 读取预处理数据 / Load preprocessed data

**输入 / Input:** 自动定位的包内 0.009H 点云和对应 k=7 法向。点、法向逐行对应，样本编号来自包内样本表。使用 prepared 模式，不重复体素化和求法向。

Bundled 0.009H clouds and matching k=7 normals; rows correspond and sample IDs come from the catalog. Prepared mode does not repeat voxelization or normal estimation.

**输出 / Output:** `data.samples`（43 个样本，每个含 N×3 点和法向）及全部样本点云总览。图中最多抽取 8000 点仅供显示，后续分析使用全部输入点。

43 sample objects with N×3 points/normals and an all-sample cloud atlas. At most 8000 points are displayed per cloud; analysis uses every input point.

In [ ]:
# 中文：输入与输出路径来自自动启动模块；直接加载已处理点云和法向。
# English: use automatically resolved paths and load existing matching arrays.
data = preprocess(environment["input_dir"], output_dir=environment["output_dir"], input_mode="prepared")


## 2. NVPD 构建 / Construct NVPDs

**输入 / Input:** 每个样本的全部法向 / All normals per sample.

将法向映射为极角 θ（与 +z 的夹角）和方位角 φ（从 +x 朝 +y），每个法向投一票，以 1° 分箱后按总票数归一化。

Map normals to polar angle θ from +z and azimuth φ from +x toward +y, count one vote per normal in 1° bins, then normalize by the total count.

**输出 / Output:** `nvpd.raw`，43×180×360；各样本概率和为1。自动显示全部 43 个原始 NVPD。

`nvpd.raw`, shape 43×180×360, unit probability per sample; all 43 raw NVPDs are displayed. Each bin is probability, not density per unit spherical area.

In [ ]:
# 中文：一个模块函数完成映射、分箱、归一化和全部样本绘图。
# English: one module handles mapping, binning, normalization, and the complete atlas.
nvpd = compute_nvpd(data)


## 3. 峰值对齐 / Align facade peaks

**输入 / Input:** 未平滑的原始 NVPD / Original unsmoothed NVPDs.

累加 θ∈[80°,100°) 的概率，用五列循环均值找方位最大峰，把整张矩阵移动使该峰到0°。这建立共同方向参考，不表示物理立面被严格对齐。找峰用的五列均值不改变原矩阵。

Sum probabilities over θ∈[80°,100°), find the maximum azimuthal peak using a circular five-column average, and shift the entire original matrix to place the peak at 0°. This establishes a reference without proving exact physical facade alignment. The peak-search average does not smooth the original matrix.

**输出 / Output:** `aligned.aligned` 与 `aligned.anchors`；显示全部对齐 NVPD、各样本原方位曲线/找峰曲线及选中峰。

Aligned matrices, prealignment anchors in degrees, all-sample atlas, and peak-search diagnostics.

In [ ]:
# 中文：先对齐；保留原始矩阵以及每个样本的角度锚点。
# English: align first; preserve the original matrices and individual peak anchors.
aligned = align_nvpd(nvpd)


## 4. 方位角平滑 / Smooth azimuth

**输入 / Input:** 对齐后的 NVPD / Aligned NVPDs.

应用 1×5 循环重叠平均，偏移为 −2、−1、0、1、2 列，步长1，仅沿方位方向处理，不平滑极角、不压缩矩阵。

Apply circular overlapping 1×5 averaging at offsets −2,−1,0,1,2 with stride one, affecting azimuth only. Polar rows and matrix dimensions are preserved.

**输出 / Output:** `smoothed.smoothed`，仍为 43×180×360，概率和为1。显示全部样本；三个阶段共用概率色标，gamma=0.3只增强显示，不影响计算。

Final 43×180×360 probabilities and all-sample atlas. All three representation stages share one probability scale; gamma 0.3 affects display only.

In [ ]:
# 中文：对齐之后才平滑；此输出是 √JSD 的正式输入。
# English: smooth only after alignment; this is the input to distance computation.
smoothed = smooth_nvpd(aligned)


## 5. 样本间距离 / Compute pairwise distances

**输入 / Input:** 平滑后的单位总质量概率分布 / Smoothed unit-mass distributions.

使用底数2的 √JSD；SciPy 已返回平方根，不能再开方。43个样本共903个无序样本对。

Use base-2 √JSD; SciPy already returns the square root. There are 903 unordered pairs among 43 samples.

**输出 / Output:** `distances.distance`，43×43对称矩阵，主对角线0、值域[0,1]，并显示编号距离热图。值越大代表法向分布差异越大。

A symmetric 43×43 matrix with zero diagonal and distances in [0,1], plus a numbered heatmap. Larger values indicate more different normal distributions.

In [ ]:
# 中文：批量计算全部样本对，保留与样本表相同的数组顺序。
# English: compare every pair and preserve the recorded sample order.
distances = compute_distances(smoothed)


## 6. HCA 与分簇选择 / HCA and cluster selection

**输入 / Input:** √JSD 距离矩阵 / √JSD distance matrix.

平均链接构建树，在有效 k=2…8 切分中选择平均轮廓系数最高者。本单元格自动选择，不人为固定为3。轮廓系数越高表示相对簇内紧凑/簇间分离更好，不直接证明类型学正确。

Build average-linkage HCA and select the valid k=2…8 cut with the highest mean silhouette. The cell does not force three clusters. Higher silhouettes indicate relative compactness/separation, not architectural correctness.

**输出 / Output:** `clusters.hca`，包含树、成员标签、候选k分数与切线高度；显示编号树状图及轮廓曲线，分支颜色与切线对应同一个选定k。

Linkage tree, memberships, candidate scores and cut height, plus the dendrogram and silhouette curve. Colors and cut line use the same selected partition.

In [ ]:
# 中文：按轮廓系数自动选簇；无需手动填写分簇数。
# English: select the cut automatically by silhouette; no manual k setting.
clusters = cluster_hca(distances)


## 7. 簇特征与差异 / Cluster characteristics and differences

**输入 / Input:** 全部最终 NVPD、距离及簇标签 / Final NVPDs, distances and memberships.

簇均值使用全部成员；簇内差异使用全部无序样本对；簇间差异使用两个簇全部交叉样本对。每格 JSD 贡献 c 总和为 d²，通过 c/d 分配 √JSD 距离 d；无符号平均图总和为相应平均距离。

Cluster means use all members; within maps use all unordered within-cluster pairs; between maps use all cross-cluster pairs. Binwise JSD terms c sum to d²; c/d allocates distance d=√JSD. Unsigned mean maps sum to mean pair distances.

**输出 / Output:** `comparison.cluster_profiles` 及簇均值、簇内差异、簇间有符号贡献图。红色表示前一簇在该角格有更多概率，蓝色表示后一簇较多；白色可能由差异小或正负抵消造成，并不是“更接近哪个簇”。统计大小使用无符号图。

Means and within/between maps. Red indicates higher probability in the first cluster; blue in the second. White may mean small differences or signed cancellation, not proximity to a cluster. Use unsigned maps for magnitudes.

In [ ]:
# 中文：使用全部簇成员与全部样本对，自动输出三类解释图。
# English: use all members and all pairs; display three sets of interpretation maps.
comparison = compare_clusters(clusters)


## 8. 保存完整结果 / Save the complete result

**输入 / Input:** 上述全部模块输出 / All preceding stages.

统一保存 CSV、JSON、NPY/NPZ、140张PNG。每个样本有原始/对齐/平滑三张独立 NVPD（共129张），另有11张点云/矩阵/分析总览。图像数字编号对应下面的使用说明。

Save CSV, JSON, NPY/NPZ and 140 PNGs: three individual NVPD stages for every sample (129 images) plus 11 overview/analysis figures.

**输出 / Output:** `result` 字典；`result["output"]` 是自动创建的结果路径。所有数组共享导出的样本顺序；`summary.json` 记录参数、输入哈希、依赖版本和簇成员。

The `result` dictionary and timestamped output directory. Array order is recorded in `samples.csv`; `summary.json` records settings, hashes, versions and memberships.

In [ ]:
# 中文：自动保存到包内 results/ 的新运行目录；不用修改输出路径。
# English: save to a new run directory under results/; no output path editing.
result = save_results(comparison)


## 结果检查与解读 / Check and read the results

参考环境结果：自动选择 **k=3**，簇大小 **4、5、34**，平均轮廓系数约 **0.11778668**。这些是核对值，不是强制写入的答案；距离与分簇由当前代码从包内输入重新计算。不同兼容库版本或平台可能带来微小浮点差异，实际版本会记录。

Reference result: **k=3**, cluster sizes **4, 5, 34**, mean silhouette **0.11778668**. These are checks, not hardcoded classifications: this run computes distances and labels from bundled inputs. Compatible library/platform changes may introduce floating-point differences; versions are recorded.

| 文件 / File | 内容 / Content |
| --- | --- |
| `samples.csv` | 样本编号/名称、点数、锚点、簇号和单例轮廓系数 / IDs, names, counts, anchors, labels and individual silhouettes |
| `sqrt_jsd_matrix.csv` | 903个样本对的43×43距离表 / All-pair 43×43 distances |
| `silhouette_by_k.csv` | 候选分簇数及轮廓分数 / Candidate k and mean silhouettes |
| `linkage.csv` | HCA合并节点、距离和数量 / HCA nodes, heights and counts |
| `summary.json` | 参数、版本、哈希、簇成员、代表样本和比较统计 / Settings, versions, hashes, memberships and comparison statistics |
| `arrays/raw.npy`, `aligned.npy`, `smoothed.npy` | 三个处理阶段的完整43×180×360矩阵 / Complete representation stages |
| `arrays/cluster_profiles.npz` | 簇均值、簇内贡献、簇间有/无符号贡献 / Cluster means and within/between maps |
| `figures/01…11_*.png` | 各模块总览与分析图 / Main overview and analysis figures |
| `figures/samples/*.png` | 每个样本三个阶段的独立图像 / Three NVPD stages for each sample |

输入包内的 `samples.csv` 是数据目录；输出的同名文件增加了本次分析结果。样本编号是显示标识，不是 Python 数组索引。

The input `samples.csv` is the dataset catalog; the output file with the same name adds this run's results. Display IDs are not Python array indices.

本 notebook 使用预处理完成的数据开展全量主实验；它不重复测绘、人工清理或 OBJ 体素化。若要重新求 k=7 法向，可在了解说明后将第一模块改为 voxel_points；默认复现无需任何修改。

The notebook starts from completed preprocessing, without repeating acquisition, manual cleaning or OBJ voxelization. Re-estimating k=7 normals is an optional advanced use of voxel_points mode; the default reproduction requires no edits.
